# 04 — Classical ML Models (scikit-learn)

Train and evaluate SVM, Random Forest, LDA, and Gradient Boosting on extracted EEG features.

**What this notebook covers:**
- Loading the processed feature matrix
- Training multiple classifiers
- Cross-validation comparison
- Confusion matrices and ROC curves
- Feature importance (Random Forest)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.features.extractor import EEGFeatureExtractor
from src.models.classical import EEGClassicalClassifier
from src.utils.evaluation import plot_confusion_matrix, compare_models

%matplotlib inline
TASK = 'sleep'   # 'focus' | 'stress' | 'sleep'
TASK_CLASSES = {
    'focus':  ['Relaxed', 'Focused'],
    'stress': ['No Stress', 'Stress'],
    'sleep':  ['Wake', 'N1', 'N2', 'N3', 'REM'],
}
CLASS_NAMES = TASK_CLASSES[TASK]
print(f'Task: {TASK}, classes: {CLASS_NAMES}')

## 1. Load & Extract Features

In [ ]:
epochs = np.load(f'../data/processed/{TASK}_data.npy')
labels = np.load(f'../data/processed/{TASK}_labels.npy')

print(f'Epochs: {epochs.shape}  |  Labels: {labels.shape}')

sfreq_map = {'focus': 160, 'stress': 128, 'sleep': 100}
extractor = EEGFeatureExtractor(sfreq=sfreq_map[TASK])
X = extractor.transform(epochs)
y = labels

print(f'Feature matrix: {X.shape}')

## 2. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 3. Train & Cross-Validate All Models

In [ ]:
MODELS = ['lda', 'svm', 'random_forest', 'gradient_boosting']
results = {}

for model_name in MODELS:
    print(f'\n{"─"*40}')
    clf = EEGClassicalClassifier(model_name=model_name)
    clf.train(X_train, y_train)
    cv = clf.cross_validate(X_train, y_train, cv=5)
    res = clf.evaluate(X_test, y_test, target_names=CLASS_NAMES)
    results[model_name] = res

## 4. Model Comparison

In [ ]:
compare_models(results, metric='accuracy')
compare_models(results, metric='f1_macro')

## 5. Best Model — Confusion Matrix

In [ ]:
best_name = max(results, key=lambda k: results[k]['accuracy'])
print(f'Best model: {best_name}  ({results[best_name]["accuracy"]:.4f} accuracy)')

plot_confusion_matrix(
    results[best_name]['confusion_matrix'],
    class_names=CLASS_NAMES,
    title=f'{best_name.upper()} — Confusion Matrix',
    normalize=True
)

## 6. Feature Importance (Random Forest)

In [ ]:
rf_clf = EEGClassicalClassifier(model_name='random_forest')
rf_clf.train(X_train, y_train)

importances = rf_clf.pipeline.named_steps['clf'].feature_importances_
feat_names  = extractor.get_feature_names()

top_k = 20
top_idx = np.argsort(importances)[::-1][:top_k]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    [feat_names[i] for i in top_idx[::-1]],
    importances[top_idx[::-1]],
    color='#3498db'
)
ax.set_xlabel('Importance')
ax.set_title(f'Top {top_k} Features — Random Forest')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()